In [1]:
# Install required libraries
!pip -q install transformers datasets accelerate scikit-learn gradio

# Check if GPU is available
import torch
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [2]:
# Import necessary libraries
import numpy as np
from datasets import load_dataset

# Load AG News dataset from Hugging Face
dataset = load_dataset("ag_news")

# Define label names (fixed mapping in AG News)
label_names = ["World", "Sports", "Business", "Sci/Tech"]

print("Training samples:", len(dataset["train"]))
print("Testing samples:", len(dataset["test"]))
print("Example sample:", dataset["train"][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Training samples: 120000
Testing samples: 7600
Example sample: {'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [3]:
from transformers import AutoTokenizer

# Load BERT tokenizer
model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# Function to tokenize text
def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True)

# Apply tokenization to dataset
tokenized = dataset.map(tokenize_function, batched=True)

tokenized


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

In [4]:
# To speed up training, we use a subset of the dataset
small_train = tokenized["train"].shuffle(seed=42).select(range(20000))
small_test  = tokenized["test"].shuffle(seed=42).select(range(2000))

print("Subset training size:", len(small_train))
print("Subset testing size:", len(small_test))


Subset training size: 20000
Subset testing size: 2000


In [5]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

# Load BERT model for classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=4  # 4 categories in AG News
)

# Data collator handles dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Model loaded successfully.")


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully.


In [6]:
from sklearn.metrics import accuracy_score, f1_score

# Define custom evaluation function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }


In [7]:
from transformers import TrainingArguments, Trainer

# Define training parameters
training_args = TrainingArguments(
    output_dir="bert-agnews",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,      # 1 epoch for faster demo
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

# Create Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready.")


Trainer ready.


In [8]:
# Start fine-tuning
trainer.train()


Step,Training Loss
50,1.019945
100,0.510331
150,0.359354
200,0.354747
250,0.362154


Step,Training Loss
50,1.019945
100,0.510331
150,0.359354
200,0.354747
250,0.362154
300,0.380784
350,0.306269
400,0.283951
450,0.299473
500,0.303997


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.3123170356750488, metrics={'train_runtime': 399.7309, 'train_samples_per_second': 50.034, 'train_steps_per_second': 3.127, 'total_flos': 1003737366857472.0, 'train_loss': 0.3123170356750488, 'epoch': 1.0})

In [9]:
# Evaluate model performance
results = trainer.evaluate()
results


{'eval_loss': 0.261600136756897,
 'eval_accuracy': 0.918,
 'eval_f1_macro': 0.9189905307482216,
 'eval_runtime': 10.9297,
 'eval_samples_per_second': 182.987,
 'eval_steps_per_second': 11.437,
 'epoch': 1.0}

In [10]:
# Save trained model and tokenizer
model.save_pretrained("bert-agnews-best")
tokenizer.save_pretrained("bert-agnews-best")

print("Model saved successfully.")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully.


In [11]:
# Save the trained model locally

model.save_pretrained("./bert-agnews-best")
tokenizer.save_pretrained("./bert-agnews-best")

print("Model saved locally.")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved locally.


In [13]:
import os

print(os.listdir("./bert-agnews-best"))


['tokenizer.json', 'tokenizer_config.json', 'model.safetensors', 'config.json']


In [14]:
from transformers import pipeline

label_names = ["World", "Sports", "Business", "Sci/Tech"]

clf = pipeline(
    "text-classification",
    model="./bert-agnews-best",
    tokenizer="./bert-agnews-best"
)

def predict_topic(text):
    result = clf(text)[0]
    label_id = int(result["label"].split("_")[-1])
    return label_names[label_id], float(result["score"])

print(predict_topic("Apple releases new AI technology."))
print(predict_topic("The football team won the championship match."))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

('Sci/Tech', 0.9760441780090332)
('Sports', 0.9882545471191406)


In [15]:
from transformers import pipeline

label_names = ["World", "Sports", "Business", "Sci/Tech"]

clf = pipeline(
    "text-classification",
    model="./bert-agnews-best",
    tokenizer="./bert-agnews-best"
)

def predict_topic(text):
    result = clf(text)[0]  # {'label': 'LABEL_2', 'score': ...}
    label_id = int(result["label"].split("_")[-1])
    return label_names[label_id], float(result["score"])

print(predict_topic("Apple releases new AI technology for iPhone and Mac."))
print(predict_topic("The team won the cricket match after a thrilling final over."))
print(predict_topic("Oil prices rise as global markets react to new policies."))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

('Sci/Tech', 0.9724556803703308)
('Sports', 0.9747551083564758)
('Business', 0.9814943075180054)


In [16]:
!pip -q install gradio
import gradio as gr

def gradio_predict(text):
    label, conf = predict_topic(text)
    return f"Predicted Topic: {label}\nConfidence: {conf:.4f}"

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(lines=3, placeholder="Type a news headline or short news text..."),
    outputs="text",
    title="AG News Topic Classifier (BERT)",
    description="Predicts one of 4 topics: World, Sports, Business, Sci/Tech."
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7af447d4d54e0c15b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Observations
- The model successfully learns to classify news text into four categories.
- Fine-tuning BERT improves performance compared to simple ML baselines.
- Some headlines can be ambiguous, which may reduce confidence scores.

## Conclusion
In this task, a BERT-based classifier was fine-tuned on the AG News dataset.
The model was evaluated using accuracy and macro F1-score and deployed using Gradio
for live interaction. This demonstrates an end-to-end NLP workflow using Transformers.
